# Intermediate feature perturbation study

The assignment PDF labels this item as **[Optional]** on page 2, but this notebook lets you run it if you want it in the report.

In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import traceback
import assignment2_gpu_split_module as mod

# Force a reload of the module in case you edited the .py file
importlib.reload(mod)

# Hardcoded root to the folder where your previous 12-pair runs saved outputs
TRAINING_ROOT = Path(
    "/data/Sajjan_Singh/cv_assign_04/"
    "cv_assign_04_gpu_split_local_fast5/"
    "cv_assign_04_gpu_split_local_fast5/"
    "cv_assign_04"
).resolve()

# Patch the module's internal ROOT variable to point to our local path
mod.ROOT = TRAINING_ROOT

# Create the standard project structure if any folder is missing
for name in ["tables", "plots", "checkpoints", "logs", "reports", "slides", "data", "downloads", "splits", "configs"]:
    (mod.ROOT / name).mkdir(parents=True, exist_ok=True)

# Data config: using existing local ImageNet100 files
IMAGENET100_MODE = "local_existing"
IMAGENET100_ROOT = mod.ROOT / "data" / "imagenet100"
TRAIN_PRETRAINED = False

# List of Dataset + Model pairs to run the ablation on
PAIR_LIST = [
    ("cifar10", "resnet"),
    ("fashion_mnist", "resnet"),
    ("imagenet100", "resnet"),
]

# Quick sanity check prints
print("Patched ROOT:", mod.ROOT)
print("ImageNet root:", IMAGENET100_ROOT)
# Check if the specific function we need is actually in the imported module
print("Has feature perturbation function:", hasattr(mod, "run_feature_perturbation_ablation"))

2026-03-21 21:28:27.772705: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 21:28:27.862179: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
Device: cuda
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA H100 NVL
cifar10              -> https://www.cs.toronto.edu/~kriz/cifar.html
fashion_mnist        -> https://github.com/zalandoresearch/fashion-mnist
imagenet100_hf       -> https://huggingface.co/datasets/clane9/imagenet-100
imagenet100_kaggle   -> https://www.kaggle.com/datasets/ambityga/imagenet100
imagecorruptions     -> https://github.com/bethgelab/imagecorruptions
robustness_repo      -> https://github.com/hendrycks/robustness


- input size: `224`
- policy K values: `1, 3, 5, 10, 15`
- backbones: `vgg16_bn, resnet50, convnext_tiny, vit_base_patch16_224`

Project root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
Device: cuda
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA H100 NVL
cifar10              -> https://www.cs.toronto.edu/~kriz/cifar.html
fashion_mnist        -> https://github.com/zalandoresearch/fashion-mnist
imagenet100_hf       -> https://huggingface.co/datasets/clane9/imagenet-100
imagenet100_kaggle   -> https://www.kaggle.com/datasets/ambityga/imagenet100
imagecorruptions     -> https://github.com/bethgelab/imagecorruptions
robustness_repo      -> https://github.com/hendrycks/robustness


- input size: `224`
- policy K values: `1, 3, 5, 10, 15`
- backbones: `vgg16_bn, resnet50, convnext_tiny, vit_base_patch16_224`

Patched ROOT: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
ImageNet root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/data/imagenet100
Has feature perturbation function: True


In [ ]:
all_feat_rows = []
error_rows = []

# Loop through all experimental pairs
for dataset_name, model_name in PAIR_LIST:
    print("\n" + "=" * 100)
    print(f"Feature perturbation study: {dataset_name} + {model_name}")
    print("=" * 100)

    try:
        # Try running the heavy computation
        df = mod.run_feature_perturbation_ablation(
            dataset_name=dataset_name,
            model_name=model_name,
            imagenet100_source=IMAGENET100_MODE,
            imagenet100_root=IMAGENET100_ROOT,
            train_pretrained=TRAIN_PRETRAINED,
        )
        # Tag results with metadata
        df["dataset"] = dataset_name
        df["model"] = model_name
        all_feat_rows.append(df)

    except Exception as e:
        # If it fails, capture the full error (traceback) so we can debug later
        tb = traceback.format_exc()
        print("Error while running feature perturbation study:")
        print(tb)
        error_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "error": str(e),
            "traceback": tb,
        })

# Hard stop if everything failed—indicates a path or module issue
if len(all_feat_rows) == 0:
    raise RuntimeError("No feature perturbation results were produced. Check the error log CSV.")

# Merge successful runs and save to CSV
feature_perturbation_master = pd.concat(all_feat_rows, ignore_index=True)
feature_perturbation_master.to_csv(mod.ROOT / "tables" / "feature_perturbation_master.csv", index=False)

# Log errors to a separate file so we don't lose track of what failed
error_df = pd.DataFrame(error_rows)
error_df.to_csv(mod.ROOT / "tables" / "feature_perturbation_errors.csv", index=False)

# Quick look at the top 50 rows and any errors
display(feature_perturbation_master.head(50))
display(error_df.head(20))

# Print final save locations for the paper's table generation
print(mod.ROOT / "tables" / "feature_perturbation_master.csv")
print(mod.ROOT / "tables" / "feature_perturbation_errors.csv")


Feature perturbation study: cifar10 + resnet
torch.compile enabled for this run.
Epoch 1/8 | training...
Epoch 1/8 | clean validation...
Epoch 1/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: brightness
  validating corruption: jpeg_compression
{'epoch': 1, 'train_loss': 2.1651049224853516, 'train_acc': 0.2272, 'lr': 0.009619397662556433, 'time_sec': 54.29982805252075, 'val_acc__clean': 0.2893, 'val_acc__gaussian_noise': 0.2515, 'val_acc__motion_blur': 0.2886, 'val_acc__fog': 0.1765, 'val_acc__brightness': 0.2628, 'val_acc__jpeg_compression': 0.2906, 'val_acc__corr_k1_s2': 0.2515, 'val_acc__corr_k3_s2': 0.23886666666666667, 'val_acc__corr_k5_s2': 0.254}
Epoch 2/8 | training...
Epoch 2/8 | clean validation...
Epoch 2/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  valida

,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,cifar10,resnet,corr_k3_s2,corrupt,6,0.301533,3,"gaussian_noise, motion_blur, fog",2,0.4789,0.35746,0.12144,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,cifar10,resnet,corr_k3_s2,corrupt,8,0.303333,3,"gaussian_noise, motion_blur, fog",2,0.4808,0.35476,0.12604,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,cifar10,resnet,corr_k5_s2,corrupt,8,0.355000,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.4700,0.35580,0.11420,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


Saved bonus ablation to: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_cifar10_resnet.csv

Feature perturbation study: fashion_mnist + resnet
torch.compile enabled for this run.
Epoch 1/8 | training...
Epoch 1/8 | clean validation...


W0321 22:18:01.999000 990679 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] torch._dynamo hit config.cache_size_limit (8)
W0321 22:18:01.999000 990679 site-packages/torch/_dynamo/convert_frame.py:906] [0/8]    function: 'forward' (/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/timm/models/resnet.py:773)
W0321 22:18:01.999000 990679 site-packages/torch/_dynamo/convert_frame.py:906] [0/8]    last reason: 0/5: Cache line invalidated because L['self']._modules['fc']._forward_hooks[list(L['self']._modules['fc']._forward_hooks.keys())[0]] got deallocated
W0321 22:18:01.999000 990679 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0321 22:18:01.999000 990679 site-packages/torch/_dynamo/convert_frame.py:906] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Epoch 1/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: brightness
  validating corruption: jpeg_compression
{'epoch': 1, 'train_loss': 1.5184824126561483, 'train_acc': 0.43666666666666665, 'lr': 0.009619397662556433, 'time_sec': 29.739888429641724, 'val_acc__clean': 0.48683333333333334, 'val_acc__gaussian_noise': 0.29991666666666666, 'val_acc__motion_blur': 0.46975, 'val_acc__fog': 0.10125, 'val_acc__brightness': 0.12158333333333333, 'val_acc__jpeg_compression': 0.4716666666666667, 'val_acc__corr_k1_s2': 0.29991666666666666, 'val_acc__corr_k3_s2': 0.2903055555555556, 'val_acc__corr_k5_s2': 0.2928333333333334}
Epoch 2/8 | training...
Epoch 2/8 | clean validation...
Epoch 2/8 | corrupted validation over 5 corruption types...
  validating corruption: gaussian_noise
  validating corruption: motion_blur
  validating corruption: fog
  validating corruption: 

...

,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082322,3,"gaussian_noise, motion_blur, fog",2,0.198308,0.114954,0.083354,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082023,3,"gaussian_noise, motion_blur, fog",2,0.199000,0.114723,0.084277,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,imagenet100,resnet,corr_k3_s2,corrupt,6,0.080969,3,"gaussian_noise, motion_blur, fog",2,0.199077,0.114277,0.084800,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


Saved bonus ablation to: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_imagenet100_resnet.csv


,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path,where
0,cifar10,resnet,corr_k3_s2,corrupt,6,0.301533,3,"gaussian_noise, motion_blur, fog",2,0.478900,0.357460,0.121440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
1,cifar10,resnet,corr_k3_s2,corrupt,8,0.303333,3,"gaussian_noise, motion_blur, fog",2,0.480800,0.354760,0.126040,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
2,cifar10,resnet,corr_k5_s2,corrupt,8,0.355000,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.470000,0.355800,0.114200,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late
3,fashion_mnist,resnet,corr_k3_s2,corrupt,5,0.374972,3,"gaussian_noise, motion_blur, fog",2,0.557200,0.383380,0.173820,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
4,fashion_mnist,resnet,corr_k3_s2,corrupt,8,0.412083,3,"gaussian_noise, motion_blur, fog",2,0.775200,0.443280,0.331920,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
5,fashion_mnist,resnet,corr_k3_s2,corrupt,8,0.410944,3,"gaussian_noise, motion_blur, fog",2,0.776400,0.442900,0.333500,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late
6,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082322,3,"gaussian_noise, motion_blur, fog",2,0.198308,0.114954,0.083354,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,early
7,imagenet100,resnet,corr_k3_s2,corrupt,6,0.082023,3,"gaussian_noise, motion_blur, fog",2,0.199000,0.114723,0.084277,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,mid
8,imagenet100,resnet,corr_k3_s2,corrupt,6,0.080969,3,"gaussian_noise, motion_blur, fog",2,0.199077,0.114277,0.084800,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,late


""


/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_master.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/feature_perturbation_errors.csv
